In [49]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------- -------------------------------- 2.4/12.8 MB 24.2 MB/s eta 0:00:01
     ----------------------------------- --- 11.8/12.8 MB 38.1 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 36.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [50]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
import numpy as np
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [35]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [36]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [37]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [38]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [39]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [56]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [57]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti (osebe, kraji, jeziki, narodi)
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    # ustvarimo množico besed, ki so del prepoznanih entitet (imen)
    imena_v_tekstu = {ent.text.lower() for ent in doc.ents if ent.label_ in odstrani_entitete}

    for token in doc:
        # če je beseda ime (NER)
        if token.text.lower() in imena_v_tekstu:
            continue
            
        # spaCy uporablja oznake: NOUN ADJ
        if token.pos_ in ['NOUN', 'ADJ']:
            beseda = token.lemma_.lower()
            
            # samo črke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [52]:
import random
from sklearn.feature_extraction import text

In [58]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    'book', 'novel', 'story', 'reader', 'edition', 'classic', 'introduction',
    'publish', 'note', 'cover', 'series', 'time', 'year', 'new', 'make', 'tell',
    'begin', 'just', 'work', 'face', 'place', 'mean', 'text', 'author', 'original', 
    'u', 'seller', 'masterpiece', 'literature', 'best', 'read', 'man', 'men', 
    'woman', 'life', 'fiction', 'tale', 'character', 'page', 'write', 'writer',
    'chapter', 'volume', 'collection', 'everything', 'day', 'world', 'come', 'series',
    'know', 'want', 'come', 'dragomir', 'tessa', 'dimitri', 'strigoi','lissa', 'sit', 'say', 'isbn',
    'bestselle', 'review'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [61]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)

#random.seed(42)
# vzamemo 150/200 knjig, ostale bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\leto_2003\03_ang_opisi\*.txt')[:150]
#filepaths= random.sample(filepaths, 150)
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
# custom_stopwords = list(text.ENGLISH_STOP_WORDS.union({'book', 'novel', 'story', 'author', 'character',
#                                                   'edition', 'classic', 'bestseller', 'review',
#                                                   'read', 'reader', 'york', 'times', 'new'}))
vectorizer= CountVectorizer(stop_words= all_stopwords, #custom_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.9)

In [62]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [63]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3314 stored elements and shape (150, 595)>
  Coords	Values
  (0, 173)	1
  (0, 133)	2
  (0, 434)	1
  (0, 359)	1
  (0, 376)	1
  (0, 443)	1
  (0, 395)	1
  (0, 549)	1
  (0, 380)	1
  (0, 406)	1
  (0, 224)	1
  (0, 199)	2
  (0, 503)	1
  (0, 396)	1
  (0, 108)	1
  (0, 501)	1
  (0, 126)	1
  (0, 332)	1
  (0, 209)	1
  (0, 1)	1
  (0, 116)	1
  (0, 525)	1
  (0, 125)	1
  (0, 393)	1
  (0, 312)	1
  :	:
  (147, 89)	1
  (147, 69)	3
  (147, 551)	1
  (147, 255)	1
  (147, 360)	1
  (147, 56)	1
  (147, 293)	1
  (147, 354)	1
  (147, 478)	1
  (148, 359)	1
  (148, 376)	1
  (148, 462)	1
  (148, 403)	1
  (148, 75)	1
  (149, 380)	1
  (149, 108)	1
  (149, 358)	1
  (149, 237)	1
  (149, 350)	1
  (149, 256)	1
  (149, 451)	1
  (149, 120)	1
  (149, 593)	2
  (149, 59)	1
  (149, 185)	2
[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 2 0]]
   account  action  actual  addition  adolescence  adventure  

In [64]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [ ]:
# test 

W, H, errors = nmf(X, 5)
print(errors)
#print(W)
#print(H)

[np.float64(70.51921256209505), np.float64(70.01334673528025), np.float64(69.64286692020751), np.float64(69.25101521112585), np.float64(68.91499210099681), np.float64(68.6852897388741), np.float64(68.52637129717449), np.float64(68.40521210528058), np.float64(68.30675769819969), np.float64(68.22155416478134), np.float64(68.1461815814595), np.float64(68.0795046093686), np.float64(68.02138893940243), np.float64(67.97114841205801), np.float64(67.92776248271663), np.float64(67.89032321659927), np.float64(67.8574391935689), np.float64(67.82827130402038), np.float64(67.80242630737847), np.float64(67.77949018479693), np.float64(67.75920727320583), np.float64(67.74129905961307), np.float64(67.72540301954463), np.float64(67.71102433719803), np.float64(67.69755044508814), np.float64(67.6844465042757), np.float64(67.67145618055406), np.float64(67.65871207925824), np.float64(67.6463822203308), np.float64(67.63452620865849), np.float64(67.62287976176619), np.float64(67.61126213284385), np.float64(67

In [66]:
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[-15:]]
    print(f"Tema {topic_idx +1}: {' '.join(top_words)}")

Tema 1: heart home night wife prison city beautiful dark marriage power thing old secret family love
Tema 2: violence savage compelling realm body experience narrative bone way religious brother violent fundamentalist birth faith
Tema 3: bloody city extraordinary people great vivid love camp country boy powerful empire historical war history
Tema 4: voice spirit friend mind love boy professional heart school medical young family girl child mother
